# Simulación de Eliminación Directa y Torneo Completo - Mundial 2026

Simulación de la fase de eliminación directa del Mundial 2026 y del torneo completo (end-to-end) utilizando distribuciones de Poisson basadas en ratings Elo.  

### Características del Modelo:
1. **Fase de Grupos**: Simulación de todos los partidos para obtener los 2 mejores de cada grupo y los 8 mejores terceros.
2. **Backtracking de Terceros**: Algoritmo recursivo inteligente para emparejar a los mejores terceros en los Dieciseisavos (Round of 32) bajo el estricto reglamento FIFA (evitando que jueguen contra rivales de su propio grupo y respetando los líderes permitidos).
3. **Resolución de Empates en Eliminación Directa**:
   * **Prórroga**: Simulación de 30 minutos extras con Poisson utilizando lambda escalado por 1/3 (estilo `sim_goles.ipynb`).
   * **Tanda de Penaltis**: Tiro a tiro con probabilidad base del 75% si persiste la igualdad en la prórroga.
4. **Fixture Oficial FIFA**: Todos los cruces y numeración de partidos exactos del fixture oficial (Partidos 73 al 104).
5. **Análisis Monte Carlo**: 1,000 corridas completas del torneo guardando probabilidades a `output/tournament_probabilities.csv`.
6. **Visualizaciones Premium**: Gráficos de favoritos en barra y curvas de progreso/supervivencia con Plotly.

In [65]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
from collections import defaultdict, Counter
from scipy.stats import poisson

print("Librerías importadas correctamente")

Librerías importadas correctamente


## Cargar ratings de los equipos

Cargamos los ratings Elo oficiales desde el archivo `output/ratings_championship.csv`.

In [66]:
ratings_df = pd.read_csv('output/fifa_elo.csv')
ratings_dict = dict(zip(ratings_df['Equipo'], ratings_df['Rating']))
print(f"Ratings cargados: {len(ratings_dict)} equipos")
ratings_df.head(10)

Ratings cargados: 235 equipos


,Equipo,Rating
0,Spain,2165
1,Argentina,2113
2,France,2081
3,England,2020
4,Brazil,1984
5,Portugal,1984
6,Colombia,1975
7,Netherlands,1961
8,Ecuador,1933
9,Croatia,1930


## Funciones de Simulación Base (Poisson, Prórroga y Penaltis)

Definimos la lógica de simulación de partidos reglamentarios, tiempo extra Poisson (escalado a 1/3 del partido de 90 minutos) y penales de desempate.

In [67]:
def get_team_rating(team_name, ratings_df):
    """Obtener el rating de un equipo"""
    rating = ratings_df[ratings_df['Equipo'] == team_name]['Rating']
    if len(rating) == 0:
        raise ValueError(f"Equipo no encontrado: {team_name}")
    return rating.values[0]

def calculate_lambda(home_team, away_team, ratings_df):
    """Calcular lambda (goles esperados) para cada equipo"""
    home_rating = get_team_rating(home_team, ratings_df)
    away_rating = get_team_rating(away_team, ratings_df)
    rating_diff = home_rating - away_rating

    lambda_home = 1.35 + (rating_diff / 200)
    lambda_away = 1.35 - (rating_diff / 300)

    lambda_home = max(0.2, lambda_home)
    lambda_away = max(0.2, lambda_away)

    return lambda_home, lambda_away

def simulate_match(home_team, away_team, ratings_df):
    """Simular un partido (90 minutos) usando Poisson"""
    lambda_home, lambda_away = calculate_lambda(home_team, away_team, ratings_df)
    goals_home = np.random.poisson(lambda_home)
    goals_away = np.random.poisson(lambda_away)
    return goals_home, goals_away

def simulate_knockout_match(home_team, away_team, ratings_df):
    """Simular un partido de eliminación directa con prórroga y penales en caso de empate"""
    # 1. 90 minutos reglamentarios
    gh, ga = simulate_match(home_team, away_team, ratings_df)
    score_90 = f"{gh}-{ga}"
    
    if gh != ga:
        return gh, ga, score_90, False, False, (home_team if gh > ga else away_team)
    
    # 2. Prórroga (Tiempo Extra de 30 minutos, se escala lambda por 1/3 al estilo sim_goles.ipynb)
    lambda_home, lambda_away = calculate_lambda(home_team, away_team, ratings_df)
    lambda_home_et = lambda_home / 3.0
    lambda_away_et = lambda_away / 3.0
    
    gh_et = np.random.poisson(lambda_home_et)
    ga_et = np.random.poisson(lambda_away_et)
    
    gh_total = gh + gh_et
    ga_total = ga + ga_et
    score_et = f"{gh_total}-{ga_total} (pro.)"
    
    if gh_total != ga_total:
        return gh_total, ga_total, score_et, True, False, (home_team if gh_total > ga_total else away_team)
    
    # 3. Tanda de penaltis (Lanzamientos de penalties con 75% éxito base)
    p_success = 0.75
    pen_home = 0
    pen_away = 0
    
    # 5 tiros iniciales por equipo
    for _ in range(5):
        if np.random.random() < p_success: 
            pen_home += 1
        if np.random.random() < p_success: 
            pen_away += 1
            
    # Si siguen empatados, muerte súbita (tiro a tiro)
    if pen_home == pen_away:
        while True:
            shot_home = np.random.random() < p_success
            shot_away = np.random.random() < p_success
            if shot_home: 
                pen_home += 1
            if shot_away: 
                pen_away += 1
            if shot_home != shot_away:
                break
                
    score_pen = f"{gh_total}-{ga_total} ({pen_home}-{pen_away} pen.)"
    winner = home_team if pen_home > pen_away else away_team
    return gh_total, ga_total, score_pen, True, True, winner

print("Funciones de simulación de partidos definidas")

Funciones de simulación de partidos definidas


## Simulación de la Fase de Grupos y Clasificación de Terceros

Simulamos todos los partidos de cada grupo y clasificamos a los mejores terceros basándonos en puntos, diferencia de gol, goles a favor y rating Elo.

In [68]:
def simulate_group(teams, ratings_df, group_name):
    """Simular fase de grupos"""
    table = defaultdict(lambda: {'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0, 'w': 0, 'd': 0, 'l': 0})
    matches = []

    for i, home in enumerate(teams):
        for j, away in enumerate(teams):
            if i >= j:
                continue

            gh, ga = simulate_match(home, away, ratings_df)

            matches.append({
                'group': group_name, 'home': home, 'away': away,
                'home_goals': gh, 'away_goals': ga, 'score': f"{gh}-{ga}"
            })

            table[home]['gf'] += gh
            table[home]['ga'] += ga
            table[away]['gf'] += ga
            table[away]['ga'] += gh

            if gh > ga:
                table[home]['pts'] += 3
                table[home]['w'] += 1
                table[away]['l'] += 1
            elif gh < ga:
                table[away]['pts'] += 3
                table[away]['w'] += 1
                table[home]['l'] += 1
            else:
                table[home]['pts'] += 1
                table[away]['pts'] += 1
                table[home]['d'] += 1
                table[away]['d'] += 1

    for team in table:
        table[team]['gd'] = table[team]['gf'] - table[team]['ga']

    results = []
    for team in teams:
        results.append({
            'group': group_name, 'team': team, 'pts': table[team]['pts'],
            'w': table[team]['w'], 'd': table[team]['d'], 'l': table[team]['l'],
            'gf': table[team]['gf'], 'ga': table[team]['ga'], 'gd': table[team]['gd']
        })

    df = pd.DataFrame(results)
    df = df.sort_values(['pts', 'gd', 'gf'], ascending=[False, False, False])
    df['position'] = range(1, 5)
    return df, pd.DataFrame(matches)

def get_best_third_placed_teams(final_table, ratings_df):
    """Obtener los 8 mejores terceros utilizando Elo rating como desempate final"""
    thirds_df = final_table[final_table['position'] == 3].copy()
    thirds_df['rating'] = thirds_df['team'].apply(lambda x: get_team_rating(x, ratings_df))
    thirds_df = thirds_df.sort_values(['pts', 'gd', 'gf', 'rating'], ascending=[False, False, False, False])
    return thirds_df.head(8)

print("Funciones de fase de grupos y clasificación de terceros definidas")

Funciones de fase de grupos y clasificación de terceros definidas


## Algoritmo de Emparejamiento por Backtracking para Mejores Terceros

Utilizamos un algoritmo recursivo inteligente para resolver la matriz de emparejamiento de los 8 mejores terceros en Dieciseisavos (Round of 32) de acuerdo a las limitaciones oficiales de la FIFA.

In [69]:
def assign_thirds_to_winners(qualified_thirds_groups, allowed_rules):
    """Algoritmo recursivo con backtracking para emparejar mejores terceros"""
    winners = ['A', 'B', 'D', 'E', 'G', 'I', 'K', 'L']
    
    def backtrack(w_idx, available_thirds, current_assignment):
        if w_idx == len(winners):
            return current_assignment
        
        winner = winners[w_idx]
        allowed = allowed_rules[winner]
        
        for group in allowed:
            if group in available_thirds:
                new_available = list(available_thirds)
                new_available.remove(group)
                
                new_assignment = current_assignment.copy()
                new_assignment[winner] = group
                
                res = backtrack(w_idx + 1, new_available, new_assignment)
                if res is not None:
                    return res
        return None

    return backtrack(0, qualified_thirds_groups, {})

print("Algoritmo de backtracking para mejores terceros definido")

Algoritmo de backtracking para mejores terceros definido


## Fixture de Eliminación Directa de la FIFA (Partidos 73 a 104)

Definimos el bracket oficial con la numeración y cruces exactos provistos por el fixture de la FIFA.

In [70]:
def simulate_world_cup_knockouts(group_winners, group_runners_up, qualified_thirds_by_group, ratings_df):
    """Simular fase de eliminación directa completa"""
    allowed_rules = {
        'A': ['C', 'E', 'F', 'H', 'I'],
        'B': ['E', 'F', 'G', 'I', 'J'],
        'D': ['B', 'E', 'F', 'I', 'J'],
        'E': ['A', 'B', 'C', 'D', 'F'],
        'G': ['A', 'E', 'H', 'I', 'J'],
        'I': ['C', 'D', 'F', 'G', 'H'],
        'K': ['D', 'E', 'I', 'J', 'L'],
        'L': ['E', 'H', 'I', 'J', 'K']
    }
    
    # Emparejar terceros
    thirds_groups_list = sorted(list(qualified_thirds_by_group.keys()))
    thirds_assignment = assign_thirds_to_winners(thirds_groups_list, allowed_rules)
    
    if thirds_assignment is None:
        thirds_assignment = {}
        winners = ['A', 'B', 'D', 'E', 'G', 'I', 'K', 'L']
        for i, w in enumerate(winners):
            thirds_assignment[w] = thirds_groups_list[i % len(thirds_groups_list)]
            
    third_opponents = {w: qualified_thirds_by_group[t_group] for w, t_group in thirds_assignment.items()}
    
    # --- Dieciseisavos de Final (Partidos 73 a 88) ---
    r32_matches = {
        73: (group_runners_up['A'], group_runners_up['B']),
        74: (group_winners['E'], third_opponents['E']),
        75: (group_winners['F'], group_runners_up['C']),
        76: (group_winners['C'], group_runners_up['F']),
        77: (group_winners['I'], third_opponents['I']),
        78: (group_runners_up['E'], group_runners_up['I']),
        79: (group_winners['A'], third_opponents['A']),
        80: (group_winners['L'], third_opponents['L']),
        81: (group_winners['D'], third_opponents['D']),
        82: (group_winners['G'], third_opponents['G']),
        83: (group_runners_up['K'], group_runners_up['L']),
        84: (group_winners['H'], group_runners_up['J']),
        85: (group_winners['B'], third_opponents['B']),
        86: (group_winners['J'], group_runners_up['H']),
        87: (group_winners['K'], third_opponents['K']),
        88: (group_runners_up['D'], group_runners_up['G'])
    }
    
    r32_winners = {}
    r32_results = []
    for match_id, (home, away) in r32_matches.items():
        gh, ga, score_str, et, pen, winner = simulate_knockout_match(home, away, ratings_df)
        r32_winners[match_id] = winner
        r32_results.append({
            'match_id': match_id, 'stage': 'R32', 'home': home, 'away': away,
            'home_goals': gh, 'away_goals': ga, 'score': score_str,
            'extra_time': et, 'penalties': pen, 'winner': winner
        })
        
    # --- Octavos de Final (Partidos 89 a 96) ---
    r16_matches = {
        89: (r32_winners[74], r32_winners[77]),
        90: (r32_winners[73], r32_winners[75]),
        91: (r32_winners[76], r32_winners[78]),
        92: (r32_winners[79], r32_winners[80]),
        93: (r32_winners[83], r32_winners[84]),
        94: (r32_winners[81], r32_winners[82]),
        95: (r32_winners[86], r32_winners[88]),
        96: (r32_winners[85], r32_winners[87])
    }
    
    r16_winners = {}
    r16_results = []
    for match_id, (home, away) in r16_matches.items():
        gh, ga, score_str, et, pen, winner = simulate_knockout_match(home, away, ratings_df)
        r16_winners[match_id] = winner
        r16_results.append({
            'match_id': match_id, 'stage': 'R16', 'home': home, 'away': away,
            'home_goals': gh, 'away_goals': ga, 'score': score_str,
            'extra_time': et, 'penalties': pen, 'winner': winner
        })
        
    # --- Cuartos de Final (Partidos 97 a 100) ---
    qf_matches = {
        97: (r16_winners[89], r16_winners[90]),
        98: (r16_winners[93], r16_winners[94]),
        99: (r16_winners[91], r16_winners[92]),
        100: (r16_winners[95], r16_winners[96])
    }
    
    qf_winners = {}
    qf_losers = {}
    qf_results = []
    for match_id, (home, away) in qf_matches.items():
        gh, ga, score_str, et, pen, winner = simulate_knockout_match(home, away, ratings_df)
        qf_winners[match_id] = winner
        qf_losers[match_id] = away if winner == home else home
        qf_results.append({
            'match_id': match_id, 'stage': 'QF', 'home': home, 'away': away,
            'home_goals': gh, 'away_goals': ga, 'score': score_str,
            'extra_time': et, 'penalties': pen, 'winner': winner
        })
        
    # --- Semifinales (Partidos 101 y 102) ---
    sf_matches = {
        101: (qf_winners[97], qf_winners[98]),
        102: (qf_winners[99], qf_winners[100])
    }
    
    sf_winners = {}
    sf_losers = {}
    sf_results = []
    for match_id, (home, away) in sf_matches.items():
        gh, ga, score_str, et, pen, winner = simulate_knockout_match(home, away, ratings_df)
        sf_winners[match_id] = winner
        sf_losers[match_id] = away if winner == home else home
        sf_results.append({
            'match_id': match_id, 'stage': 'SF', 'home': home, 'away': away,
            'home_goals': gh, 'away_goals': ga, 'score': score_str,
            'extra_time': et, 'penalties': pen, 'winner': winner
        })
        
    # --- Tercer Puesto (Partido 103) ---
    home_3rd, away_3rd = sf_losers[101], sf_losers[102]
    gh_3rd, ga_3rd, score_3rd, et_3rd, pen_3rd, winner_3rd = simulate_knockout_match(home_3rd, away_3rd, ratings_df)
    loser_3rd = away_3rd if winner_3rd == home_3rd else home_3rd
    
    third_place_result = {
        'match_id': 103, 'stage': '3RD_PLACE', 'home': home_3rd, 'away': away_3rd,
        'home_goals': gh_3rd, 'away_goals': ga_3rd, 'score': score_3rd,
        'extra_time': et_3rd, 'penalties': pen_3rd, 'winner': winner_3rd
    }
    
    # --- Gran Final (Partido 104) ---
    home_f, away_f = sf_winners[101], sf_winners[102]
    gh_f, ga_f, score_f, et_f, pen_f, winner_f = simulate_knockout_match(home_f, away_f, ratings_df)
    runner_up = away_f if winner_f == home_f else home_f
    
    final_result = {
        'match_id': 104, 'stage': 'FINAL', 'home': home_f, 'away': away_f,
        'home_goals': gh_f, 'away_goals': ga_f, 'score': score_f,
        'extra_time': et_f, 'penalties': pen_f, 'winner': winner_f
    }
    
    return {
        'champion': winner_f,
        'runner_up': runner_up,
        'third': winner_3rd,
        'fourth': loser_3rd,
        'r32_winners': list(r32_winners.values()),
        'r16_winners': list(r16_winners.values()),
        'qf_winners': list(qf_winners.values()),
        'sf_winners': list(sf_winners.values()),
        'matches': r32_results + r16_results + qf_results + sf_results + [third_place_result, final_result]
    }

print("Fixture de eliminación directa y brackets oficiales definidos")

Fixture de eliminación directa y brackets oficiales definidos


## Simulación de Torneo Completo (Grupos + Eliminación Directa)

Integramos la fase de grupos y la eliminatoria para correr simulaciones completas.

In [71]:
def simulate_full_tournament(groups, ratings_df):
    """Simular un torneo completo: grupos y eliminación directa"""
    all_group_results = []
    group_winners = {}
    group_runners_up = {}
    thirds_by_group = {}
    
    for group_name, teams in groups.items():
        df, _ = simulate_group(teams, ratings_df, group_name)
        all_group_results.append(df)
        
        group_winners[group_name] = df[df['position'] == 1]['team'].values[0]
        group_runners_up[group_name] = df[df['position'] == 2]['team'].values[0]
        thirds_by_group[group_name] = df[df['position'] == 3]['team'].values[0]
        
    final_table = pd.concat(all_group_results, ignore_index=True)
    best_thirds = get_best_third_placed_teams(final_table, ratings_df)
    
    qualified_thirds_by_group = {row['group']: thirds_by_group[row['group']] for _, row in best_thirds.iterrows()}
        
    knockout_summary = simulate_world_cup_knockouts(
        group_winners, group_runners_up, qualified_thirds_by_group, ratings_df
    )
    return final_table, knockout_summary

print("Función de simulación del torneo completo definida")

Función de simulación del torneo completo definida


## Definir Grupos del Mundial 2026

Definición oficial de grupos personalizados en base a `simulacion_fase_grupos.ipynb`.

In [72]:
custom_groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama']
}
print("Grupos del Mundial 2026 definidos")

Grupos del Mundial 2026 definidos


## Simulación Única de Prueba (Demo)

Corremos una simulación completa única del Mundial 2026 y mostramos los resultados del bracket eliminatorio oficial.

In [100]:
np.random.seed()
final_table, knockout = simulate_full_tournament(custom_groups, ratings_df)

print("==================================================")
print("DEMO: SIMULACIÓN ÚNICA END-TO-END DEL MUNDIAL 2026")
print("==================================================")
print(f"🏆 CAMPEÓN: {knockout['champion']} 🏆")
print(f"🥈 Subcampeón: {knockout['runner_up']}")
print(f"🥉 Tercer Puesto: {knockout['third']}")
print(f"4️⃣ Cuarto Puesto: {knockout['fourth']}")
print("==================================================\n")

stage_names = {
    'R32': 'Dieciseisavos de Final (Round of 32)',
    'R16': 'Octavos de Final (Round of 16)',
    'QF': 'Cuartos de Final',
    'SF': 'Semifinales',
    '3RD_PLACE': 'Definición del Tercer Puesto',
    'FINAL': 'Gran Final'
}

curr_stage = ""
for m in knockout['matches']:
    if m['stage'] != curr_stage:
        curr_stage = m['stage']
        print(f"\n--- {stage_names[curr_stage]} ---")
    print(f"Partido {m['match_id']}: {m['home']} vs {m['away']} | Marcador: {m['score']} -> Ganador: {m['winner']}")

DEMO: SIMULACIÓN ÚNICA END-TO-END DEL MUNDIAL 2026
🏆 CAMPEÓN: Spain 🏆
🥈 Subcampeón: Argentina
🥉 Tercer Puesto: France
4️⃣ Cuarto Puesto: Netherlands


--- Dieciseisavos de Final (Round of 32) ---
Partido 73: Mexico vs Bosnia and Herzegovina | Marcador: 1-0 -> Ganador: Mexico
Partido 74: Germany vs South Korea | Marcador: 4-2 -> Ganador: Germany
Partido 75: Japan vs Morocco | Marcador: 2-0 -> Ganador: Japan
Partido 76: Brazil vs Netherlands | Marcador: 0-1 -> Ganador: Netherlands
Partido 77: France vs Egypt | Marcador: 3-0 -> Ganador: France
Partido 78: Ecuador vs Norway | Marcador: 0-0 (4-3 pen.) -> Ganador: Ecuador
Partido 79: Czech Republic vs Scotland | Marcador: 2-2 (5-4 pen.) -> Ganador: Czech Republic
Partido 80: Croatia vs Uzbekistan | Marcador: 3-2 -> Ganador: Croatia
Partido 81: Turkey vs Switzerland | Marcador: 2-2 (5-2 pen.) -> Ganador: Turkey
Partido 82: Belgium vs Senegal | Marcador: 3-1 -> Ganador: Belgium
Partido 83: Colombia vs England | Marcador: 1-0 -> Ganador: Colomb

## Simulación Monte Carlo de Torneo Completo

Corremos la simulación completa del torneo 1,000 veces y calculamos la probabilidad acumulada de cada selección de alcanzar cada fase del fixture.

In [74]:
def run_tournament_monte_carlo(groups, ratings_df, n_sims=100):
    stats = defaultdict(lambda: {
        'group_stage': 0, 'r32': 0, 'r16': 0, 'qf': 0,
        'fourth': 0, 'third': 0, 'runner_up': 0, 'champion': 0
    })
    
    print(f"Iniciando Monte Carlo con {n_sims} simulaciones...")
    print("=" * 50)
    
    for sim in range(n_sims):
        if (sim + 1) % 250 == 0 or sim == 0 or sim + 1 == n_sims:
            print(f"Simulando torneo {sim + 1}/{n_sims}...")
            
        final_table, knockout = simulate_full_tournament(groups, ratings_df)
        
        # Participantes de Dieciseisavos
        r32_teams = set()
        for m in knockout['matches']:
            if m['stage'] == 'R32':
                r32_teams.add(m['home'])
                r32_teams.add(m['away'])
                
        # Registrar eliminados en grupos
        all_teams = [t for grp, t_list in groups.items() for t in t_list]
        for team in all_teams:
            if team not in r32_teams: 
                stats[team]['group_stage'] += 1
                
        # Quedaron en R32
        r16_teams = set(knockout['r16_winners'])
        for team in r32_teams:
            if team not in r16_teams: 
                stats[team]['r32'] += 1
                
        # Quedaron en R16
        qf_teams = set(knockout['qf_winners'])
        for team in r16_teams:
            if team not in qf_teams: 
                stats[team]['r16'] += 1
                
        # Quedaron en QF
        sf_teams = set(knockout['sf_winners'])
        for team in qf_teams:
            if team not in sf_teams: 
                stats[team]['qf'] += 1
                
        stats[knockout['champion']]['champion'] += 1
        stats[knockout['runner_up']]['runner_up'] += 1
        stats[knockout['third']]['third'] += 1
        stats[knockout['fourth']]['fourth'] += 1
        
    print("=" * 50)
    print("¡Simulaciones completadas!")
    
    records = []
    for team, counts in stats.items():
        c, ru, t3, f4, qf, r16, r32 = counts['champion'], counts['runner_up'], counts['third'], counts['fourth'], counts['qf'], counts['r16'], counts['r32']
        
        prob_champion = c / n_sims * 100
        prob_final = (c + ru) / n_sims * 100
        prob_sf = (c + ru + t3 + f4) / n_sims * 100
        prob_qf = (c + ru + t3 + f4 + qf) / n_sims * 100
        prob_r16 = (c + ru + t3 + f4 + qf + r16) / n_sims * 100
        prob_r32 = (c + ru + t3 + f4 + qf + r16 + r32) / n_sims * 100
        
        records.append({
            'team': team,
            'rating': get_team_rating(team, ratings_df),
            'Dieciseisavos (R32) %': prob_r32,
            'Octavos (R16) %': prob_r16,
            'Cuartos (QF) %': prob_qf,
            'Semifinal (SF) %': prob_sf,
            'Final (F) %': prob_final,
            'Campeón %': prob_champion
        })

    df_results = pd.DataFrame(records)
    df_results = df_results.sort_values('Campeón %', ascending=False).reset_index(drop=True)
    return df_results

# Ejecutar 1,000 simulaciones para la demo
n_sims = 1000
prob_df = run_tournament_monte_carlo(custom_groups, ratings_df, n_sims=n_sims)
prob_df.to_csv('output/tournament_probabilities.csv', index=False)
print("Probabilidades guardadas en 'output/tournament_probabilities.csv'")
prob_df.head(20).round(1)

Iniciando Monte Carlo con 1000 simulaciones...
Simulando torneo 1/1000...
Simulando torneo 250/1000...
Simulando torneo 500/1000...
Simulando torneo 750/1000...
Simulando torneo 1000/1000...
¡Simulaciones completadas!
Probabilidades guardadas en 'output/tournament_probabilities.csv'


,team,rating,Dieciseisavos (R32) %,Octavos (R16) %,Cuartos (QF) %,Semifinal (SF) %,Final (F) %,Campeón %
0,Spain,2165,115.7,98.0,92.6,76.9,61.2,43.9
1,Argentina,2113,114.8,92.4,77.8,63.0,48.2,25.3
2,France,2081,134.0,111.7,96.5,62.2,27.9,13.2
3,England,2020,121.0,82.2,58.8,37.8,16.8,5.7
4,Brazil,1984,114.2,55.4,36.2,21.9,7.6,2.8
5,Portugal,1984,106.2,49.0,22.8,15.4,8.0,2.4
6,Colombia,1975,108.3,45.6,23.8,14.8,5.8,1.4
7,Netherlands,1961,115.6,64.6,37.1,20.9,4.7,1.3
8,Ecuador,1933,109.4,35.9,22.5,12.5,2.5,0.9
9,Turkey,1902,98.2,40.2,11.5,6.8,2.1,0.6


## Visualización de Resultados con Plotly

Diseñamos gráficos interactivos de alta calidad para visualizar las probabilidades de salir campeón y las curvas de supervivencia/progreso.

In [75]:
# 1. Gráfico de Favoritos a Campeón
top_15 = prob_df.head(15)
fig_fav = px.bar(
    top_15, x='Campeón %', y='team', orientation='h',
    title='Top 15 Favoritos a Ganar el Mundial 2026 (%)',
    labels={'Campeón %': 'Probabilidad de ganar (%)', 'team': 'Equipo'},
    color='Campeón %', color_continuous_scale='Blues',
    text='Campeón %'
)
fig_fav.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_fav.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig_fav.show()

# 2. Gráfico de Supervivencia / Progresión de Contendientes
contenders = ['Spain', 'France', 'Argentina', 'Brazil', 'England', 'Germany', 'Portugal']
df_contenders = prob_df[prob_df['team'].isin(contenders)].copy()

stages = ['Dieciseisavos (R32) %', 'Octavos (R16) %', 'Cuartos (QF) %', 'Semifinal (SF) %', 'Final (F) %', 'Campeón %']
stage_labels = ['Dieciseisavos (R32)', 'Octavos (R16)', 'Cuartos (QF)', 'Semifinal', 'Final', 'Campeón']

fig_survival = go.Figure()
for _, row in df_contenders.iterrows():
    team_probs = [row[s] for s in stages]
    fig_survival.add_trace(go.Scatter(
        x=stage_labels, y=team_probs, mode='lines+markers', name=row['team'],
        line=dict(width=3), marker=dict(size=8)
))

fig_survival.update_layout(
    title='Curva de Progreso y Supervivencia en el Mundial 2026 (Principales Contendientes)',
    xaxis_title='Fase del Torneo',
    yaxis_title='Probabilidad de alcanzar la fase (%)',
    yaxis=dict(range=[0, 105]),
    height=500
)
fig_survival.show()

In [76]:
final_table

,group,team,pts,w,d,l,gf,ga,gd,position
0,A,Mexico,9,3,0,0,10,1,9,1
1,A,Czech Republic,6,2,0,1,3,5,-2,2
2,A,South Korea,3,1,0,2,3,4,-1,3
3,A,South Africa,0,0,0,3,1,7,-6,4
4,B,Switzerland,7,2,1,0,3,1,2,1
5,B,Canada,6,2,0,1,3,1,2,2
6,B,Bosnia and Herzegovina,3,1,0,2,4,4,0,3
7,B,Qatar,1,0,1,2,1,5,-4,4
8,C,Morocco,7,2,1,0,12,4,8,1
9,C,Brazil,7,2,1,0,11,3,8,2
